In [ ]:
import json, pandas as pd, numpy as np
rows=[json.loads(l) for l in open("../../data/ciphers_pilot.jsonl")]
df=pd.DataFrame(rows)
CAP=df["n_turns"].max()
for col in ["first_action_turn","first_explicit_turn","first_production_turn"]:
    df[col+"_c"]=df[col].fillna(CAP+1)   # censored: never = cap+1
# Contamination tag (spec 6): keyed/arbitrary ciphers require real inference; the rest are
# standard transformations the model has likely seen in training (measure memory, not
# inference). Keep them separable so the article never conflates the two.
NOVEL={"random_substitution","block_permutation"}
df["kind"]=np.where(df["cipher"].isin(NOVEL),"novel","known")
df.head()

In [ ]:
# Difficulty ranking. Censored mean ALONE is misleading: a cipher no model ever cracks
# (all cells = cap+1) looks almost identical to one cracked at the very last turn. Report
# the comprehension RATE (fraction of cells ever solved) and the median turn among SOLVED
# cells alongside the censored mean, so "impossible" and "barely solved" are distinguishable.
def summarize(g):
    solved=g["first_action_turn"].notna()
    return pd.Series({
        "n": len(g),
        "comprehension_rate": solved.mean(),
        "median_turn_if_solved": g.loc[solved,"first_action_turn"].median(),
        "mean_censored": g["first_action_turn_c"].mean(),
    })
rank=(df.groupby("cipher").apply(summarize)
        .sort_values(["comprehension_rate","median_turn_if_solved"],ascending=[True,False]))
print("hardest (lowest comprehension rate) first:\n")
print(rank)

In [ ]:
import matplotlib.pyplot as plt
piv=df.pivot_table(index="cipher",columns="model",values="first_action_turn_c",aggfunc="mean")
fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(piv.values,aspect="auto",cmap="viridis_r")
ax.set_xticks(range(len(piv.columns)));ax.set_xticklabels(piv.columns,rotation=45,ha="right")
ax.set_yticks(range(len(piv.index)));ax.set_yticklabels(piv.index)
fig.colorbar(im,label="mean turns to comprehension (cap+1 = never)")
fig.savefig("../../data/ciphers_comprehension_heatmap.png",dpi=140,bbox_inches="tight")

In [ ]:
# Comprehension vs production, by protocol and by contamination kind.
by_proto=df.groupby("protocol")[["first_action_turn_c","first_production_turn_c"]].mean()
print("by protocol (censored means):\n", by_proto, "\n")
# known vs novel: does real inference (novel) lag memorized ciphers, and does the few-shot
# plaintext key help novel ciphers more than known ones? (report rate, not just censored mean)
by_kind=df.groupby(["kind","protocol"]).agg(
    comprehension_rate=("first_action_turn", lambda s: s.notna().mean()),
    mean_action_censored=("first_action_turn_c","mean"),
    mean_production_censored=("first_production_turn_c","mean"),
)
print("by kind x protocol:\n", by_kind)